In [0]:
%pip install faker
# Update these to match the catalog and schema
# that you used for the pipeline in step 1.
catalog = "playground"
schema = dbName = db = "cdc-pipeline"

spark.sql(f'USE CATALOG `{catalog}`')
spark.sql(f'USE SCHEMA `{schema}`')
spark.sql(f'CREATE VOLUME IF NOT EXISTS `{catalog}`.`{db}`.`raw_data`')
volume_folder =  f"/Volumes/{catalog}/{db}/raw_data"

try:
  dbutils.fs.ls(volume_folder+"/customers")
except:
  print(f"folder doesn't exist, generating the data under {volume_folder}...")
  from pyspark.sql import functions as F
  from faker import Faker
  from collections import OrderedDict
  import uuid
  fake = Faker()
  import random

  fake_firstname = F.udf(fake.first_name)
  fake_lastname = F.udf(fake.last_name)
  fake_email = F.udf(fake.ascii_company_email)
  fake_date = F.udf(lambda:fake.date_time_this_month().strftime("%m-%d-%Y %H:%M:%S"))
  fake_address = F.udf(fake.address)
  operations = OrderedDict([("APPEND", 0.5),("DELETE", 0.1),("UPDATE", 0.3),(None, 0.01)])
  fake_operation = F.udf(lambda:fake.random_elements(elements=operations, length=1)[0])
  fake_id = F.udf(lambda: str(uuid.uuid4()) if random.uniform(0, 1) < 0.98 else None)

  df = spark.range(0, 100000).repartition(100)
  df = df.withColumn("id", fake_id())
  df = df.withColumn("firstname", fake_firstname())
  df = df.withColumn("lastname", fake_lastname())
  df = df.withColumn("email", fake_email())
  df = df.withColumn("address", fake_address())
  df = df.withColumn("operation", fake_operation())
  df_customers = df.withColumn("operation_date", fake_date())
  df_customers.repartition(100).write.format("json").mode("overwrite").save(volume_folder+"/customers")

In [0]:
# Update these to match the catalog and schema
# that you used for the pipeline in step 1.
catalog = "playground"
schema = "cdc-pipeline"

display(spark.read.json(f"/Volumes/{catalog}/{schema}/raw_data/customers"))

In [0]:
# Update these to match the catalog and schema
# that you used for the pipeline in step 1.
catalog = "playground"
schema = "cdc-pipeline"
id = "95787f4b-4f5e-4555-8480-e4879f059921"

display(spark.read.json(f"/Volumes/{catalog}/{schema}/raw_data/customers").filter(f"id = '{id}'"));

In [0]:
%sql
SELECT *
FROM `playground`.`cdc-pipeline`.`customers`
WHERE id='95787f4b-4f5e-4555-8480-e4879f059921';

In [0]:
%sql
select id, count(*) as count
FROM `playground`.`cdc-pipeline`.`customers_history`
group by id
having count > 1;

In [0]:
%sql
SELECT *
FROM `playground`.`cdc-pipeline`.`customers_history`
WHERE id='95787f4b-4f5e-4555-8480-e4879f059921';

In [0]:
%sql
SELECT *
FROM `playground`.`cdc-pipeline`.`customers_cdc_clean`
WHERE id='95787f4b-4f5e-4555-8480-e4879f059921';

In [0]:
%sql
SELECT *
FROM `playground`.`cdc-pipeline`.`customers_history_agg`
WHERE id='95787f4b-4f5e-4555-8480-e4879f059921';

In [0]:
# UPDATE RECORD
# Update these to match the catalog and schema
# that you used for the pipeline in step 1.
catalog = "playground"
schema = dbName = db = "cdc-pipeline"

spark.sql(f'USE CATALOG `{catalog}`')
spark.sql(f'USE SCHEMA `{schema}`')
spark.sql(f'CREATE VOLUME IF NOT EXISTS `{catalog}`.`{db}`.`raw_data`')
volume_folder =  f"/Volumes/{catalog}/{db}/raw_data"

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from faker import Faker
from datetime import datetime
from collections import OrderedDict

fake = Faker()
import random

# Add 1 new customer record
new_customer_data = [(
id,
fake.first_name(),
fake.last_name(),
fake.ascii_company_email(),
fake.address(),
"UPDATE",
datetime.now().strftime("%m-%d-%Y %H:%M:%S")
)]

schema = StructType([
StructField("id", StringType(), True),
StructField("firstname", StringType(), True),
StructField("lastname", StringType(), True),
StructField("email", StringType(), True),
StructField("address", StringType(), True),
StructField("operation", StringType(), True),
StructField("operation_date", StringType(), True)
])

new_customer_df = spark.createDataFrame(new_customer_data, schema=schema)
new_customer_df.write.format("json").mode("append").save(volume_folder+"/customers")
print("New customer record added successfully!")

In [0]:
# DELETE RECORD
# Update these to match the catalog and schema
# that you used for the pipeline in step 1.
catalog = "playground"
schema = dbName = db = "cdc-pipeline"

spark.sql(f'USE CATALOG `{catalog}`')
spark.sql(f'USE SCHEMA `{schema}`')
spark.sql(f'CREATE VOLUME IF NOT EXISTS `{catalog}`.`{db}`.`raw_data`')
volume_folder =  f"/Volumes/{catalog}/{db}/raw_data"

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from faker import Faker
from datetime import datetime
from collections import OrderedDict

fake = Faker()
import random
id = "95787f4b-4f5e-4555-8480-e4879f059921"
# Delete 1 customer record
new_customer_data = [(
id,
fake.first_name(),
fake.last_name(),
fake.ascii_company_email(),
fake.address(),
"DELETE",
datetime.now().strftime("%m-%d-%Y %H:%M:%S")
)]

schema = StructType([
StructField("id", StringType(), True),
StructField("firstname", StringType(), True),
StructField("lastname", StringType(), True),
StructField("email", StringType(), True),
StructField("address", StringType(), True),
StructField("operation", StringType(), True),
StructField("operation_date", StringType(), True)
])

new_customer_df = spark.createDataFrame(new_customer_data, schema=schema)
new_customer_df.write.format("json").mode("append").save(volume_folder+"/customers")
print("New customer record deleted successfully!")

In [0]:
# APPEND RECORD
catalog = "playground"
schema = dbName = db = "cdc-pipeline"

spark.sql(f'USE CATALOG `{catalog}`')
spark.sql(f'USE SCHEMA `{schema}`')
spark.sql(f'CREATE VOLUME IF NOT EXISTS `{catalog}`.`{db}`.`raw_data`')
volume_folder =  f"/Volumes/{catalog}/{db}/raw_data"

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from faker import Faker
from datetime import datetime
from collections import OrderedDict
import uuid

fake = Faker()
import random

id = "95787f4b-4f5e-4555-8480-e4879f059921"
new_customer_data = [(
  id,
  fake.first_name(),
  fake.last_name(),
  fake.ascii_company_email(),
  fake.address(),
  "APPEND",
  datetime.now().strftime("%m-%d-%Y %H:%M:%S")
)]

schema = StructType([
  StructField("id", StringType(), True),
  StructField("firstname", StringType(), True),
  StructField("lastname", StringType(), True),
  StructField("email", StringType(), True),
  StructField("address", StringType(), True),
  StructField("operation", StringType(), True),
  StructField("operation_date", StringType(), True)
])

new_customer_df = spark.createDataFrame(new_customer_data, schema=schema)
new_customer_df.write.format("json").mode("append").save(volume_folder+"/customers")
print("New APPEND record added successfully!")
print(new_customer_data)